<a href="https://colab.research.google.com/github/DaniloDuque/logistic-regression/blob/main/src/tp2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TP2 — Regresión Logística

## Setup

In [ ]:
import os, sys

if 'google.colab' in sys.modules:
    if not os.path.exists('logistic-regression'):
        os.system('git clone https://github.com/DaniloDuque/logistic-regression.git')
    os.chdir('logistic-regression')

sys.path.insert(0, os.path.abspath('src'))

---
# Sección 1 — Algoritmo de Regresión Logística

## 1.a — Usar el código del perceptrón como base
DOCUMENTACION EN LATEX: Explique en el reporte cada una de las modificaciones necesarias al código del perceptrón, utilizando como referencia las diferencias entre ambos algoritmos.

### Pruebas unitarias
2 pruebas unitarias (con su objetivo, entradas, salidas esperadas y resultados) para las funciones modificadas en el algoritmo del perceptron base.

---
## 1.b — Experimentos: Datos separables vs no separables

### Imports

In [ ]:
# Install spacy
!pip install spacy

# Download the Spanish model
!python -m spacy download es_core_news_lg

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import spacy
from pathlib import Path

from data_generator import generate_data
from logistic_regression import LogisticRegression
from trainer import train_with_history, compute_mae
from metrics import run_experiment, print_single_result, print_runs_table, compute_accuracy
from visualization import plot_results

STEPS = 1000
ALPHA = 0.1

FIGURES_DIR = Path(__file__).parent.parent / 'figures' if '__file__' in dir() else Path('..') / 'figures'
FIGURES_DIR.mkdir(exist_ok=True)


### Experimento 1 — Datos linealmente separables

Dos clases generadas con `cluster_std=1.0` (grupos compactos y bien separados).
Se espera que el modelo converja rápidamente y obtenga un MAE bajo.


In [ ]:
X_train_s, X_test_s, y_train_s, y_test_s = generate_data(separable=True, n_samples=500, random_state=42)

model_s = LogisticRegression(torch.zeros(X_train_s.shape[1], 1))
train_errors_s, test_errors_s = train_with_history(
    model_s, X_train_s, y_train_s, X_test_s, y_test_s, steps=STEPS, alpha=ALPHA
)

mae_train_s = compute_mae(y_train_s, model_s.forward(X_train_s))
mae_test_s  = compute_mae(y_test_s,  model_s.forward(X_test_s))
acc_train_s = compute_accuracy(model_s, X_train_s, y_train_s)
acc_test_s  = compute_accuracy(model_s, X_test_s,  y_test_s)

print(f"Iteraciones: {STEPS}")
print_single_result('Linealmente separable', mae_train_s, mae_test_s, acc_train_s, acc_test_s)

plot_results(model_s, X_train_s, y_train_s, train_errors_s, test_errors_s,
             'Experimento 1 — Datos linealmente separables',
             output_path=FIGURES_DIR / 'experimento1.pdf')


### Experimento 2 — Datos no linealmente separables

Dos clases generadas con `cluster_std=4.0` (grupos solapados).
Se espera que el modelo tenga dificultades y obtenga un MAE más alto.


In [ ]:
X_train_ns, X_test_ns, y_train_ns, y_test_ns = generate_data(separable=False, n_samples=500, random_state=42)

model_ns = LogisticRegression(torch.zeros(X_train_ns.shape[1], 1))
train_errors_ns, test_errors_ns = train_with_history(
    model_ns, X_train_ns, y_train_ns, X_test_ns, y_test_ns, steps=STEPS, alpha=ALPHA
)

mae_train_ns = compute_mae(y_train_ns, model_ns.forward(X_train_ns))
mae_test_ns  = compute_mae(y_test_ns,  model_ns.forward(X_test_ns))
acc_train_ns = compute_accuracy(model_ns, X_train_ns, y_train_ns)
acc_test_ns  = compute_accuracy(model_ns, X_test_ns,  y_test_ns)

print(f"Iteraciones: {STEPS}")
print_single_result('No linealmente separable', mae_train_ns, mae_test_ns, acc_train_ns, acc_test_ns)

plot_results(model_ns, X_train_ns, y_train_ns, train_errors_ns, test_errors_ns,
             'Experimento 2 — Datos no linealmente separables',
             output_path=FIGURES_DIR / 'experimento2.pdf')


### 1.2 — 10 ejecuciones: MAE medio, precisión, mínimo y máximo


In [ ]:
results_s  = run_experiment(separable=True,  steps=STEPS, alpha=ALPHA)
results_ns = run_experiment(separable=False, steps=STEPS, alpha=ALPHA)

print_runs_table(results_s, results_ns)

---
# Sección 2 — Regresión Logística y LLMs para clasificación de textos

In [ ]:
import pandas as pd
from huggingface_hub import hf_hub_download

file_path = hf_hub_download(
    repo_id="saul1917/FEINA",
    filename="FEINA_1.xlsx",
    repo_type="dataset"
)

df = pd.read_excel(file_path)
print(df.shape)
print(df.head())
print(df.columns.tolist())

### Embeddings BERT — verificación de sanidad


In [ ]:
from transformers import AutoTokenizer, AutoModelForMaskedLM
import torch
import numpy as np

# ── Modelo 1: BERT en español (dccuchile) ─────────────────────────
tokenizer1 = AutoTokenizer.from_pretrained("dccuchile/bert-base-spanish-wwm-cased")
model1 = AutoModelForMaskedLM.from_pretrained("dccuchile/bert-base-spanish-wwm-cased")
model1.eval()

# ── Modelo 2: BERTIN - RoBERTa en español ─────────────────────────
tokenizer2 = AutoTokenizer.from_pretrained("bertin-project/bertin-roberta-base-spanish")
model2 = AutoModelForMaskedLM.from_pretrained("bertin-project/bertin-roberta-base-spanish")
model2.eval()

# ── Embedding mediante token CLS ──────────────────────────────────
def get_embedding(text, model, tokenizer):
    inputs = tokenizer(text, return_tensors="pt",
                       truncation=True, max_length=512, padding=True)
    with torch.no_grad():
        outputs = model.base_model(**inputs)
    return outputs.last_hidden_state[:, 0, :].squeeze().numpy()

# ── Verificación de sanidad ────────────────────────────────────────
test = "El índice de capitalización bursátil refleja la valoración agregada."
emb1 = get_embedding(test, model1, tokenizer1)
emb2 = get_embedding(test, model2, tokenizer2)

print(f"Modelo 1 (BERT-es) forma:   {emb1.shape}")
print(f"Modelo 2 (BERTIN) forma:    {emb2.shape}")


## 1. Regresión logística con la TFIDF
### 1. a) Etapa de preprocesamiento
- Remueve stopwords
- Remueve puntuación
- Lematización
- Lo pasa a minúsculas

In [ ]:
# PRUEBAAAA
import spacy

nlp = spacy.load("es_core_news_lg")

def preprocess_text(text):
    doc = nlp(text.lower())

    tokens = []

    for token in doc:
        # Ignorar puntuación (.is_punct), espacios (.is_space) y stopwords
        if token.is_stop or token.is_punct or token.is_space:
            continue

        lemma = token.lemma_.strip() # Se usa Lemma

        if lemma:
            tokens.append(lemma)

    return " ".join(tokens)

text = "LOS estudiantes son muy laboriosos con su proyecto de Inteligencia Artificial!!!"

print(preprocess_text(text))

# ─────────────────────────────────────────────────────────────────
# Sección 2.3 — Modelos LLM para clasificación de complejidad
# Modelos:
#   1. Recognai/zeroshot_selectra_medium  (español, ELECTRA-based)
#   2. facebook/bart-large-mnli           (multilingüe, BART-based)
# ─────────────────────────────────────────────────────────────────

In [ ]:
from transformers import pipeline
import pandas as pd
import numpy as np

# ── Carga de modelos ──────────────────────────────────────────────
print("Cargando modelo 1: zeroshot_selectra_medium ...")
clf_selectra = pipeline(
    "zero-shot-classification",
    model="Recognai/zeroshot_selectra_medium"
)

print("Cargando modelo 2: bart-large-mnli ...")
clf_bart = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)

# ── Etiquetas y templates ─────────────────────────────────────────
LABELS_ES = ["simple", "complejo"]
LABELS_EN = ["simple", "complex"]
TEMPLATE_ES = "Este texto es {}."       # requerido para modelos en español
TEMPLATE_EN = "This example is {}."

print("Modelos cargados correctamente.")